### InfinityStar Single Inference with VAR-Q Memory Profiling

This notebook performs a single video generation with InfinityStar (720p),
records CUDA memory usage for comparing different VAR-Q quantisation configs.

In [1]:
import os, sys, time, json
import torch

GPU_ID = int(os.environ.get("GPU_ID", "2"))
torch.cuda.set_device(GPU_ID)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import cv2
import numpy as np
import os.path as osp

PROJECT_ROOT = '/home/jiaji_lu/AR/VAR-Q'
# 最后一档 scale 的 QKV 落盘目录名（与 SageAttention 变体名一致，便于对照实验）
QKV_DUMP_DIR_NAME = 'sageattn_qk_int8_pv_fp16_triton'
INFINITYSTAR_ROOT = osp.join(PROJECT_ROOT, 'InfinityStar')
os.chdir(INFINITYSTAR_ROOT)
sys.path.insert(0, INFINITYSTAR_ROOT)
sys.path.insert(0, PROJECT_ROOT)

from tools.run_infinity import (
    load_tokenizer, load_transformer, load_visual_tokenizer,
    gen_one_example, save_video, transform,
)
from infinity.models.self_correction import SelfCorrection
from infinity.schedules.dynamic_resolution import (
    get_dynamic_resolution_meta, get_first_full_spatial_size_scale_index,
)
from infinity.schedules import get_encode_decode_func
from infinity.utils.arg_util import Args

############# VAR-Q Configuration File #############
CONFIG_FILE = os.environ.get(
    "CONFIG_FILE",
    osp.join(PROJECT_ROOT, "VAR_Q", "Infinity-VAR_Q-8.json"),
)

from VAR_Q.config_loader import VARQConfig
config = VARQConfig(CONFIG_FILE)
quant_config = config.get_quantization_config()

print(f"[Config] Loaded from {CONFIG_FILE}")
print(f"[Config]   enable          : {quant_config.get('enable', False)}")
print(f"[Config]   q_bits          : {quant_config.get('q_bits', 8)}")
print(f"[Config]   quant_method    : {quant_config.get('quant_method', 'G_SCALE_HEAD_DIM')}")
print(f"[Config]   qkv_format      : {quant_config.get('qkv_format', 'BHLc')}")
print(f"[Config]   rescale_qk      : {quant_config.get('rescale_qk', False)}")
print(f"[Config]   fused_kv_flash  : {quant_config.get('enable_fused_kv_flashattn', False)}")
print(f"[Config]   enable_sageattn: {quant_config.get('enable_sageattn', False)}")
print(f"[Config]   sageattn_type  : {quant_config.get('sageattn_type', 'none')}")

/home/jiaji_lu/conda/envs/varq/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/jiaji_lu/conda/envs/varq/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Config] Loaded from /home/jiaji_lu/AR/VAR-Q/VAR_Q/Infinity-VAR_Q-8.json
[Config]   enable          : False
[Config]   q_bits          : 4
[Config]   quant_method    : G_SCALE_HEAD_DIM
[Config]   qkv_format      : BHLc
[Config]   rescale_qk      : False
[Config]   fused_kv_flash  : False
[Config]   enable_sageattn: False
[Config]   sageattn_type  : sageattn


In [2]:
############# Create Args (mirrors infer_video_720p.py) #############
CHECKPOINTS_DIR = '/data/jiaji_lu/WM/infinitystar/checkpoints/InfinityStar'

args = Args()
args.pn                        = '0.90M'
args.fps                       = 16
args.video_frames              = 81
args.model_path                = osp.join(CHECKPOINTS_DIR, 'infinitystar_8b_720p_weights')
args.checkpoint_type           = 'torch_shard'
args.vae_path                  = osp.join(CHECKPOINTS_DIR, 'infinitystar_videovae.pth')
args.text_encoder_ckpt         = osp.join(CHECKPOINTS_DIR, 'text_encoder/flan-t5-xl-official/')
args.model_type                = 'infinity_qwen8b'
args.text_channels             = 2048
args.dynamic_scale_schedule    = 'infinity_elegant_clip20frames_v2'
args.bf16                      = 1
args.use_apg                   = 1
args.use_cfg                   = 0
args.cfg                       = 34
args.tau_image                 = 1
args.tau_video                 = 0.4
args.apg_norm_threshold        = 0.05
args.image_scale_repetition    = '[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]'
args.video_scale_repetition    = '[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 1, 1]'
args.append_duration2caption   = 1
args.use_two_stage_lfq         = 1
args.detail_scale_min_tokens   = 750
args.semantic_scales           = 12
args.max_repeat_times          = 10000
args.enable_rewriter           = 0

# VAR-Q quantization parameters
args.enable_quantization       = int(quant_config.get('enable', False))
args.q_bits                    = quant_config.get('q_bits', 8)
args.quant_method              = quant_config.get('quant_method', 'G_SCALE_HEAD_DIM')
args.qkv_format                = quant_config.get('qkv_format', 'BHLc')
args.rescale_qk                = int(quant_config.get('rescale_qk', False))
args.enable_fused_kv_flashattn = int(quant_config.get('enable_fused_kv_flashattn', False))
# 本 notebook：固定用 Triton int8 QK / fp16 PV 的 SageAttention，并配合 QKV dump
args.enable_sageattn = 1
args.sageattn_type = QKV_DUMP_DIR_NAME

print(f"[Args] model_type            : {args.model_type}")
print(f"[Args] model_path            : {args.model_path}")
print(f"[Args] vae_path              : {args.vae_path}")
print(f"[Args] VAR-Q enabled         : {bool(args.enable_quantization)}")
print(f"[Args] enable_sageattn       : {bool(args.enable_sageattn)}")
print(f"[Args] sageattn_type         : {args.sageattn_type}")
if args.enable_quantization:
    print(f"[Args]   q_bits             : {args.q_bits}")
    print(f"[Args]   quant_method       : {args.quant_method}")
    print(f"[Args]   qkv_format         : {args.qkv_format}")
    print(f"[Args]   rescale_qk         : {args.rescale_qk}")
    print(f"[Args]   fused_kv_flash     : {args.enable_fused_kv_flashattn}")

[Args] model_type            : infinity_qwen8b
[Args] model_path            : /data/jiaji_lu/WM/infinitystar/checkpoints/InfinityStar/infinitystar_8b_720p_weights
[Args] vae_path              : /data/jiaji_lu/WM/infinitystar/checkpoints/InfinityStar/infinitystar_videovae.pth
[Args] VAR-Q enabled         : False
[Args] enable_sageattn       : True
[Args] sageattn_type         : sageattn_qk_int8_pv_fp16_triton


In [3]:
############# Load models (InferencePipe) #############
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t0 = time.time()

text_tokenizer, text_encoder = load_tokenizer(t5_path=args.text_encoder_ckpt)
vae = load_visual_tokenizer(args)
vae = vae.float().to('cuda')
infinity = load_transformer(vae, args)
self_correction = SelfCorrection(vae, args)

video_encode, video_decode, get_visual_rope_embeds, get_scale_pack_info = (
    get_encode_decode_func(args.dynamic_scale_schedule)
)

load_time_ms = (time.time() - t0) * 1000
mem_after_load = torch.cuda.max_memory_reserved() / (1024 ** 2)
print(f"\n[Memory] After model load  : {mem_after_load:.0f} MB")
print(f"[Timer]  Model load time   : {load_time_ms:.0f} ms")

[Loading tokenizer and text encoder]


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 10.96it/s]


Load VAE from /data/jiaji_lu/WM/infinitystar/checkpoints/InfinityStar/infinitystar_videovae.pth


/home/jiaji_lu/AR/VAR-Q/InfinityStar/infinity/models/videovae/modules/quantizer/finite_scalar_quantization.py:144: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)


[Loading Infinity]
arch: qwen, self.pn: 0.90M, self.codebook_dim: 64, self.add_lvl_embeding_on_first_block: 0,             self.num_of_label_value: 2, self.rope2d_each_sa_layer: 1, self.rope2d_normalized_by_hw: 2             self.train_h_div_w_list: [np.float64(0.562), np.float64(1.0)], self.image_scale_repetition: [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], self.video_scale_repetition: [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 1, 1]
[precompute_rope4d_freqs_grid: 4d]: start


/home/jiaji_lu/AR/VAR-Q/InfinityStar/tools/run_infinity.py:222: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True, dtype=torch.bfloat16, cache_enabled=True), torch.no_grad():


using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageattn_qk_int8_pv_fp16_triton
using sageat

In [4]:
############# Prompt & inference params #############
prompt = (
    "Iron Man fights a giant slime monster in a ruined city at night, "
    "flying fast, firing repulsor blasts, cinematic explosions, "
    "dynamic camera, realistic lighting, wet streets, smoke, sparks, "
    "epic superhero movie style, photorealistic, 4K."
)
seed = 41
num_frames = 81
mapped_duration = 5

print(f"[Inference] prompt      : {prompt[:80]}...")
print(f"[Inference] seed        : {seed}")
print(f"[Inference] num_frames  : {num_frames}")
print(f"[Inference] cfg         : {args.cfg}")
print(f"[Inference] tau_image   : {args.tau_image}")
print(f"[Inference] tau_video   : {args.tau_video}")

[Inference] prompt      : Iron Man fights a giant slime monster in a ruined city at night, flying fast, fi...
[Inference] seed        : 41
[Inference] num_frames  : 81
[Inference] cfg         : 34
[Inference] tau_image   : 1
[Inference] tau_video   : 0.4


In [5]:
############# Generate video + memory profiling #############
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

dynamic_resolution_h_w, h_div_w_templates = get_dynamic_resolution_meta(
    args.dynamic_scale_schedule, args.video_frames,
)
h_div_w_template_ = h_div_w_templates[np.argmin(np.abs(h_div_w_templates - 0.571))]
scale_schedule = dynamic_resolution_h_w[h_div_w_template_][args.pn][
    'pt2scale_schedule'
][(num_frames - 1) // 4 + 1]

args.first_full_spatial_size_scale_index = (
    get_first_full_spatial_size_scale_index(scale_schedule)
)
args.tower_split_index = args.first_full_spatial_size_scale_index + 1
context_info = get_scale_pack_info(
    scale_schedule, args.first_full_spatial_size_scale_index, args,
)

tau = (
    [args.tau_image] * args.tower_split_index
    + [args.tau_video] * (len(scale_schedule) - args.tower_split_index)
)

full_prompt = prompt
full_prompt = f'{full_prompt}, Close-up on big objects, emphasize scale and detail'
if args.append_duration2caption:
    full_prompt = f'<<<t={mapped_duration}s>>>' + full_prompt

_qkv_root = osp.join(PROJECT_ROOT, QKV_DUMP_DIR_NAME)
os.makedirs(_qkv_root, exist_ok=True)
os.environ['INFINITYSTAR_QKV_DUMP_DIR'] = osp.abspath(_qkv_root)
print(f'[QKV] 最后一档 scale 全部 block 的 Q/K/V 将写入: {_qkv_root}')

t_start = time.time()
with torch.cuda.amp.autocast(enabled=True, dtype=torch.bfloat16, cache_enabled=True), torch.no_grad():
    generated_video, _ = gen_one_example(
        infinity,
        vae,
        text_tokenizer,
        text_encoder,
        full_prompt,
        negative_prompt='',
        g_seed=seed,
        gt_leak=-1,
        gt_ls_Bl=None,
        cfg_list=args.cfg,
        tau_list=tau,
        scale_schedule=scale_schedule,
        cfg_insertion_layer=[0],
        vae_type=args.vae_type,
        sampling_per_bits=1,
        enable_positive_prompt=0,
        low_vram_mode=True,
        args=args,
        get_visual_rope_embeds=get_visual_rope_embeds,
        context_info=context_info,
        noise_list=None,
    )
    if len(generated_video.shape) == 3:
        generated_video = generated_video.unsqueeze(0)

if 'INFINITYSTAR_QKV_DUMP_DIR' in os.environ:
    del os.environ['INFINITYSTAR_QKV_DUMP_DIR']

runtime_ms = (time.time() - t_start) * 1000
torch.cuda.synchronize()

peak_reserved_mb  = torch.cuda.max_memory_reserved()  / (1024 ** 2)
peak_allocated_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
current_reserved_mb  = torch.cuda.memory_reserved()   / (1024 ** 2)
current_allocated_mb = torch.cuda.memory_allocated()  / (1024 ** 2)

print(f"\n{'='*55}")
print(f"  InfinityStar Video Generation — Memory Report")
print(f"{'='*55}")
q_tag = f"Q{args.q_bits}-{args.quant_method}" if args.enable_quantization else "baseline (no quant)"
print(f"  Config           : {q_tag}")
print(f"  rescale_qk       : {bool(args.rescale_qk)}")
print(f"  fused_kv_flash   : {bool(args.enable_fused_kv_flashattn)}")
print(f"  -----------------------------------------")
print(f"  Peak reserved    : {peak_reserved_mb:,.0f} MB")
print(f"  Peak allocated   : {peak_allocated_mb:,.0f} MB")
print(f"  Current reserved : {current_reserved_mb:,.0f} MB")
print(f"  Current allocated: {current_allocated_mb:,.0f} MB")
print(f"  Runtime          : {runtime_ms:,.0f} ms")
print(f"{'='*55}")

[QKV] 最后一档 scale 全部 block 的 Q/K/V 将写入: /home/jiaji_lu/AR/VAR-Q/sageattn_qk_int8_pv_fp16_triton
prompt=<<<t=5s>>>Iron Man fights a giant slime monster in a ruined city at night, flying fast, firing repulsor blasts, cinematic explosions, dynamic camera, realistic lighting, wet streets, smoke, sparks, epic superhero movie style, photorealistic, 4K., Close-up on big objects, emphasize scale and detail


/tmp/ipykernel_917623/381514259.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True, dtype=torch.bfloat16, cache_enabled=True), torch.no_grad():
/home/jiaji_lu/AR/VAR-Q/InfinityStar/tools/run_infinity.py:134: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True, dtype=torch.bfloat16, cache_enabled=True):


cfg: [34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34], tau: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4]


100%|██████████| 86/86 [00:59<00:00,  7.59s/it]

[INFINITYSTAR_QKV_DUMP] last scale si=29 -> /home/jiaji_lu/AR/VAR-Q/sageattn_qk_int8_pv_fp16_triton


100%|██████████| 86/86 [04:37<00:00,  3.22s/it]

Decode takes 4.2s
cost: 277.91385316848755, infinity cost=277.2961962223053



  InfinityStar Video Generation — Memory Report
  Config           : baseline (no quant)
  rescale_qk       : False
  fused_kv_flash   : False
  -----------------------------------------
  Peak reserved    : 71,400 MB
  Peak allocated   : 59,734 MB
  Current reserved : 71,400 MB
  Current allocated: 20,516 MB
  Runtime          : 277,915 ms


In [6]:
############# Save video #############
save_dir = osp.join(PROJECT_ROOT, 'Benchmark', 'outputs', 'InfinityStar')
os.makedirs(save_dir, exist_ok=True)
video_path = osp.join(save_dir, f'demo-{q_tag}.mp4')
save_video(generated_video.cpu().numpy(), fps=args.fps, save_filepath=video_path)
print(f"[Save] {video_path}")

Video saved as /home/jiaji_lu/AR/VAR-Q/Benchmark/outputs/InfinityStar/demo-baseline (no quant).mp4
[Save] /home/jiaji_lu/AR/VAR-Q/Benchmark/outputs/InfinityStar/demo-baseline (no quant).mp4


In [7]:
############# Export memory report to JSON (append-friendly) #############
report_path = osp.join(PROJECT_ROOT, 'temp', 'infinitystar_memory_report.json')

record = {
    "config_file": CONFIG_FILE,
    "enable_quantization": bool(args.enable_quantization),
    "q_bits": args.q_bits if args.enable_quantization else None,
    "quant_method": args.quant_method if args.enable_quantization else None,
    "qkv_format": args.qkv_format if args.enable_quantization else None,
    "rescale_qk": bool(args.rescale_qk),
    "fused_kv_flash": bool(args.enable_fused_kv_flashattn),
    "peak_reserved_mb": round(peak_reserved_mb, 1),
    "peak_allocated_mb": round(peak_allocated_mb, 1),
    "current_reserved_mb": round(current_reserved_mb, 1),
    "current_allocated_mb": round(current_allocated_mb, 1),
    "load_time_ms": round(load_time_ms, 0),
    "runtime_ms": round(runtime_ms, 0),
    "gpu_id": GPU_ID,
    "gpu_name": torch.cuda.get_device_name(GPU_ID),
    "timestamp": time.strftime('%Y-%m-%d %H:%M:%S'),
}

records = []
if osp.exists(report_path):
    with open(report_path) as f:
        records = json.load(f)
records.append(record)
with open(report_path, 'w') as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

print(f"[Report] Appended to {report_path}  (total {len(records)} records)")
print(json.dumps(record, indent=2))

[Report] Appended to /home/jiaji_lu/AR/VAR-Q/temp/infinitystar_memory_report.json  (total 10 records)
{
  "config_file": "/home/jiaji_lu/AR/VAR-Q/VAR_Q/Infinity-VAR_Q-8.json",
  "enable_quantization": false,
  "q_bits": null,
  "quant_method": null,
  "qkv_format": null,
  "rescale_qk": false,
  "fused_kv_flash": false,
  "peak_reserved_mb": 71400.0,
  "peak_allocated_mb": 59734.3,
  "current_reserved_mb": 71400.0,
  "current_allocated_mb": 20516.3,
  "load_time_ms": 64394.0,
  "runtime_ms": 277915.0,
  "gpu_id": 2,
  "gpu_name": "NVIDIA A100 80GB PCIe",
  "timestamp": "2026-03-31 03:10:02"
}
